### Sample Data

In [1]:
#create sample documents
sample_docs = [
    """
    Machin learning fundamentals

    Machine learning is a branch of artificial intelligence that enables computers to learn from data and 
    improve their performance without being explicitly programmed. It works by using algorithms to analyze patterns, 
    make predictions, and solve problems based on the information provided. For example, machine learning is used in
    recommendation systems like suggesting movies on streaming platforms
    or products on shopping websites.
    """,

    """
    Deep learning foundimantals

    Deep learning is a specialized branch of machine learning that uses artificial neural networks
    to process large amounts of data and solve complex problems. It is designed to imitate the way the human brain works 
    by recognizing patterns and learning from experience. Deep learning is widely used in technologies such as 
    speech recognition, image classification, self-driving cars, and virtual assistants like Siri and Alexa. 
    Its ability to automatically learn features from raw data makes it more powerful than traditional machine
    learning in many advanced applications.
    """,

    """
    Natural language processing (NLP)

    Natural Language Processing (NLP) is a branch of artificial intelligence that focuses on enabling computers to
    understand, interpret, and generate human language. It combines machine learning and linguistics to help machines
    process text and speech in a meaningful way. NLP is used in many everyday applications such as chatbots,
    language translation, voice assistants, and text prediction. It helps computers analyze large amounts 
    of language data, identify patterns, and respond intelligently to users. As technology advances, NLP
    continues to improve communication between humans and machines.
    """
]
sample_docs

['\n    Machin learning fundamentals\n\n    Machine learning is a branch of artificial intelligence that enables computers to learn from data and \n    improve their performance without being explicitly programmed. It works by using algorithms to analyze patterns, \n    make predictions, and solve problems based on the information provided. For example, machine learning is used in\n    recommendation systems like suggesting movies on streaming platforms\n    or products on shopping websites.\n    ',
 '\n    Deep learning foundimantals\n\n    Deep learning is a specialized branch of machine learning that uses artificial neural networks\n    to process large amounts of data and solve complex problems. It is designed to imitate the way the human brain works \n    by recognizing patterns and learning from experience. Deep learning is widely used in technologies such as \n    speech recognition, image classification, self-driving cars, and virtual assistants like Siri and Alexa. \n    Its a

In [2]:
## save sample documents to files

import tempfile
temp_dir = tempfile.mkdtemp()

for i, doc in enumerate(sample_docs):
    with open(f"{temp_dir}/doc_{i}.txt", "w") as f:
        f.write(doc)

print(f"Sample document create in : {temp_dir}")

Sample document create in : C:\Users\STUDEN~1\AppData\Local\Temp\tmphto8lkg8


In [10]:
import tempfile
temp_dir = tempfile.mkdtemp()

for i, doc in enumerate(sample_docs):
    with open(f"doc_{i}.txt", "w") as f:
        f.write(doc)

In [6]:
temp_dir

'C:\\Users\\STUDEN~1\\AppData\\Local\\Temp\\tmp_jtcjp1s'

### 2.Document Loading

In [11]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader

#Load document from directory
loader = DirectoryLoader(
    "data",
    glob="*.txt",
    loader_cls=TextLoader,\
    loader_kwargs={'encoding': 'utf-8'}
)

documents = loader.load()

print(f"Loaded {len(documents)} documents")
print(f"\nFirst document previwe:")
print(documents[0].page_content[:200]+ "...")

Loaded 3 documents

First document previwe:

    Machin learning fundamentals

    Machine learning is a branch of artificial intelligence that enables computers to learn from data and 
    improve their performance without being explicitly pro...


### Documnet Splitting

In [14]:
#initialize text splitter
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 50,
    length_function = len,
    separators=[" "]

)
chunks=text_splitter.split_documents(documents)

print(f"Created {len(chunks)} chunks from {documents} documents")
print(f"\nchunk example;")
print(f"Content: {chunks[0].page_content[:150]}...")
print(f"Metadata: {chunks[0].metadata}")

Created 5 chunks from [Document(metadata={'source': 'data\\doc_0.txt'}, page_content='\n    Machin learning fundamentals\n\n    Machine learning is a branch of artificial intelligence that enables computers to learn from data and \n    improve their performance without being explicitly programmed. It works by using algorithms to analyze patterns, \n    make predictions, and solve problems based on the information provided. For example, machine learning is used in\n    recommendation systems like suggesting movies on streaming platforms\n    or products on shopping websites.\n    '), Document(metadata={'source': 'data\\doc_1.txt'}, page_content='\n    Deep learning foundimantals\n\n    Deep learning is a specialized branch of machine learning that uses artificial neural networks\n    to process large amounts of data and solve complex problems. It is designed to imitate the way the human brain works \n    by recognizing patterns and learning from experience. Deep learning is widely used 

### Embedding Models


In [15]:

import os
from dotenv import load_dotenv
load_dotenv()
   
os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")

In [16]:
from langchain_openai import OpenAIEmbeddings
sample_text = "Machin learning is facianting"
embeddings = OpenAIEmbeddings()
embeddings

OpenAIEmbeddings(client=<openai.resources.embeddings.Embeddings object at 0x000001D5D06BB9B0>, async_client=<openai.resources.embeddings.AsyncEmbeddings object at 0x000001D5D0B15310>, model='text-embedding-ada-002', dimensions=None, deployment='text-embedding-ada-002', openai_api_version=None, openai_api_base=None, openai_api_type=None, openai_proxy=None, embedding_ctx_length=8191, openai_api_key=SecretStr('**********'), openai_organization=None, allowed_special=None, disallowed_special=None, chunk_size=1000, max_retries=2, request_timeout=None, headers=None, tiktoken_enabled=True, tiktoken_model_name=None, show_progress_bar=False, model_kwargs={}, skip_empty=False, default_headers=None, default_query=None, retry_min_seconds=4, retry_max_seconds=20, http_client=None, http_async_client=None, check_embedding_ctx_length=True)

In [23]:
vector = embeddings.embed_query(sample_text)
print(len(vector))

1536


### Initialize the ChromaDB Vector Store And Stores the Chunks in Vector Representation


In [ ]:
## Create a chromadb vector store
from langchain_community.vectorstores import Chroma
persist_directory = "./chroma_db"

## initialize chromadb with opena ai embeddings
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=OpenAIEmbeddings(),
    persist_directory=persist_directory,
    collection_name="rag_collection"
)

print(f"Vector store created with {vectorstore._collection.count()} vectors")
print(f"Persist to {persist_directory}")


Vector store created with 5 vectors
Persist to ./chroma_db


In [28]:
query = "What is deep learning?"

similar_docs = vectorstore.similarity_search(query, k=3)
similar_docs

[Document(metadata={'source': 'data\\doc_1.txt'}, page_content='Deep learning foundimantals\n\n    Deep learning is a specialized branch of machine learning that uses artificial neural networks\n    to process large amounts of data and solve complex problems. It is designed to imitate the way the human brain works \n    by recognizing patterns and learning from experience. Deep learning is widely used in technologies such as \n    speech recognition, image classification, self-driving cars, and virtual assistants like Siri and Alexa. \n    Its ability to'),
 Document(metadata={'source': 'data\\doc_0.txt'}, page_content='Machin learning fundamentals\n\n    Machine learning is a branch of artificial intelligence that enables computers to learn from data and \n    improve their performance without being explicitly programmed. It works by using algorithms to analyze patterns, \n    make predictions, and solve problems based on the information provided. For example, machine learning is used

In [29]:
print(f"Query: {query}")
print(f"\nTop {len(similar_docs)} similar chunks:")
for i, doc in enumerate(similar_docs):
    print(f"\n--- Chunks{i+1} ---")
    print(doc.page_content[:200] + "...")
    print(f"Source: {doc.metadata.get('source', 'unknown')}")

Query: What is deep learning?

Top 3 similar chunks:

--- Chunks1 ---
Deep learning foundimantals

    Deep learning is a specialized branch of machine learning that uses artificial neural networks
    to process large amounts of data and solve complex problems. It is d...
Source: data\doc_1.txt

--- Chunks2 ---
Machin learning fundamentals

    Machine learning is a branch of artificial intelligence that enables computers to learn from data and 
    improve their performance without being explicitly programm...
Source: data\doc_0.txt

--- Chunks3 ---
like Siri and Alexa. 
    Its ability to automatically learn features from raw data makes it more powerful than traditional machine
    learning in many advanced applications....
Source: data\doc_1.txt


### Advanve Similarity Search With Scores

In [30]:
results_scores = vectorstore.similarity_search_with_score(query, k=3)
results_scores

[(Document(metadata={'source': 'data\\doc_1.txt'}, page_content='Deep learning foundimantals\n\n    Deep learning is a specialized branch of machine learning that uses artificial neural networks\n    to process large amounts of data and solve complex problems. It is designed to imitate the way the human brain works \n    by recognizing patterns and learning from experience. Deep learning is widely used in technologies such as \n    speech recognition, image classification, self-driving cars, and virtual assistants like Siri and Alexa. \n    Its ability to'),
  0.22163891792297363),
 (Document(metadata={'source': 'data\\doc_0.txt'}, page_content='Machin learning fundamentals\n\n    Machine learning is a branch of artificial intelligence that enables computers to learn from data and \n    improve their performance without being explicitly programmed. It works by using algorithms to analyze patterns, \n    make predictions, and solve problems based on the information provided. For example

### Initialize LLM, RAG Chain, Prompt Template, Query the RA system

In [32]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-5-nano")
#________or__________
# from langchain.chat_models.base import init_chat_model

# llm = init_chat_model("openai:gpt-5-nano")
# init_chat_model("groq:")
# llm
llm.invoke("What is AI")

AIMessage(content='AI, or artificial intelligence, is a field of computer science focused on making machines do tasks that usually require human intelligence. Instead of following rigid, hand-written steps for every situation, AI systems learn from data and improve their performance over time.\n\nKey ideas:\n- What it can do: perceive (see, hear), reason, learn, plan, understand language, and act to achieve goals.\n- How it learns: from data examples, feedback, and sometimes trial-and-error (reinforcement learning).\n- Not just one thing: AI includes many techniques, from simple rule-based systems to advanced neural networks.\n\nCommon types:\n- Narrow (or weak) AI: specialized to a single task (e.g., voice assistants, image recognition, spam filtering).\n- Artificial General Intelligence (AGI): a hypothetical system that could understand, learn, and apply intelligence across many tasks as well as a human. Not yet realized.\n\nCommon methods:\n- Machine learning: algorithms learn patte

### Modern RAG Chain

In [33]:
from langchain_classic.chains import create_retrieval_chain
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

In [34]:
## conver vectorestore to retrivel
retriever = vectorstore.as_retriever(
    search_kwarg={"k":3} ## Retrieve top 3 relevant chunks
)
retriever

VectorStoreRetriever(tags=['Chroma', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x000001D5CF283D70>, search_kwargs={})

In [36]:
# Creat a prompt template
system_prompt = """You are an assistant for question-answering tasks.
use the following pieces of retrived context to answer the question.
If you don't know the answer, just say that you don;t know.
use three sentences maximum and keep the answer concise.

context: {context}"""

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}")]
)

In [37]:
### create a document chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
document_chain = create_stuff_documents_chain(llm, prompt)
document_chain

RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
| ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks.\nuse the following pieces of retrived context to answer the question.\nIf you don't know the answer, just say that you don;t know.\nuse three sentences maximum and keep the answer concise.\n\ncontext: {context}"), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={})])
| ChatOpenAI(profile={'max_input_tokens': 272000, 'max_output_tokens': 128000, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': 

In [38]:
### Create the final RAG chain
from langchain_classic.chains import create_retrieval_chain
rag_chain = create_retrieval_chain(retriever, document_chain)
rag_chain

RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | VectorStoreRetriever(tags=['Chroma', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x000001D5CF283D70>, search_kwargs={}), kwargs={}, config={'run_name': 'retrieve_documents'}, config_factories=[])
})
| RunnableAssign(mapper={
    answer: RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
              context: RunnableLambda(format_docs)
            }), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
            | ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks.\nuse the following pieces of retrived context to answer the question.\nIf you don't k

In [40]:
response = rag_chain.invoke({"input":"What is Deep Learning?"})
response

{'input': 'What is Deep Learning?',
 'context': [Document(metadata={'source': 'data\\doc_1.txt'}, page_content='Deep learning foundimantals\n\n    Deep learning is a specialized branch of machine learning that uses artificial neural networks\n    to process large amounts of data and solve complex problems. It is designed to imitate the way the human brain works \n    by recognizing patterns and learning from experience. Deep learning is widely used in technologies such as \n    speech recognition, image classification, self-driving cars, and virtual assistants like Siri and Alexa. \n    Its ability to'),
  Document(metadata={'source': 'data\\doc_0.txt'}, page_content='Machin learning fundamentals\n\n    Machine learning is a branch of artificial intelligence that enables computers to learn from data and \n    improve their performance without being explicitly programmed. It works by using algorithms to analyze patterns, \n    make predictions, and solve problems based on the informatio

In [41]:
response['answer']

'Deep learning is a specialized branch of machine learning that uses artificial neural networks to process large amounts of data and solve complex problems. It is designed to imitate the way the human brain works by recognizing patterns and learning from experience. It is widely used in technologies such as speech recognition, image classification, self-driving cars, and virtual assistants like Siri and Alexa.'

In [43]:
# Functio n to query the modern rag system 
def query_rag_modern(question):
    print(f"qustion: {question}")
    print("-"*50)

    #using creat retrivel chain approach
    result = rag_chain.invoke({"input": question})

    print(f"Answer: {result['answer']}")
    print("\nRetrived Context:")
    for i, doc in enumerate(result['context']):
        print(f"\n--- Source {i+1} ---")
        print(doc.page_content[:200] + "...")

    return result

#test quries
test_question = [
    "what are the three type of machin learning?",
    " what is deep learning?",
    "what are CNNs best used for?"
    
]

for question in test_question:
    result = query_rag_modern(question)
    print("\n" + "="*80 + "\n")

qustion: what are the three type of machin learning?
--------------------------------------------------
Answer: Supervised learning: uses labeled data to train models.  
Unsupervised learning: finds patterns in unlabeled data.  
Reinforcement learning: learns by trial and error through interacting with an environment.

Retrived Context:

--- Source 1 ---
Machin learning fundamentals

    Machine learning is a branch of artificial intelligence that enables computers to learn from data and 
    improve their performance without being explicitly programm...

--- Source 2 ---
Deep learning foundimantals

    Deep learning is a specialized branch of machine learning that uses artificial neural networks
    to process large amounts of data and solve complex problems. It is d...

--- Source 3 ---
like Siri and Alexa. 
    Its ability to automatically learn features from raw data makes it more powerful than traditional machine
    learning in many advanced applications....

--- Source 4 ---
It

### Create RAG chain altervatve -using LCEL (LANGCHAIN EXPRESSIN LANGUAGE)


In [44]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel, RunnablePassthrough

In [52]:
#create cutom prompt
custom_prompt = ChatPromptTemplate.from_template("""Use the following context to answer the questuion.
if you don;t know the answer based on the context, say you don;t know.
provide specific detailes from context to support your answer.
                                                
context:
{context}

question: {question}

Answer:""")
custom_prompt

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='Use the following context to answer the questuion.\nif you don;t know the answer based on the context, say you don;t know.\nprovide specific detailes from context to support your answer.\n\ncontext:\n{context}\n\nquestion: {question}\n\nAnswer:'), additional_kwargs={})])

In [53]:
#format the output document
def format_docs(docs):
    return"\n\n".join(doc.page_content for doc in docs)

In [71]:
rag_chain_lcel=(
    {
        "context":retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | custom_prompt 
    | llm 
    | StrOutputParser()
)
rag_chain_lcel

{
  context: VectorStoreRetriever(tags=['Chroma', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x000001D5CF283D70>, search_kwargs={})
           | RunnableLambda(format_docs),
  question: RunnablePassthrough()
}
| ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='Use the following context to answer the questuion.\nif you don;t know the answer based on the context, say you don;t know.\nprovide specific detailes from context to support your answer.\n\ncontext:\n{context}\n\nquestion: {question}\n\nAnswer:'), additional_kwargs={})])
| ChatOpenAI(profile={'max_input_tokens': 272000, 'max_output_tokens': 128000, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_o

In [72]:
respons = rag_chain_lcel.invoke("What is Deep learning")
respons

'Deep learning is a specialized branch of machine learning that uses artificial neural networks to process large amounts of data and solve complex problems. It is designed to imitate the way the human brain works by recognizing patterns and learning from experience. It is widely used in technologies such as speech recognition, image classification, self-driving cars, and virtual assistants like Siri and Alexa.'

In [73]:
retriever.invoke("what is deep learning")

[Document(metadata={'source': 'data\\doc_1.txt'}, page_content='Deep learning foundimantals\n\n    Deep learning is a specialized branch of machine learning that uses artificial neural networks\n    to process large amounts of data and solve complex problems. It is designed to imitate the way the human brain works \n    by recognizing patterns and learning from experience. Deep learning is widely used in technologies such as \n    speech recognition, image classification, self-driving cars, and virtual assistants like Siri and Alexa. \n    Its ability to'),
 Document(metadata={'source': 'data\\doc_0.txt'}, page_content='Machin learning fundamentals\n\n    Machine learning is a branch of artificial intelligence that enables computers to learn from data and \n    improve their performance without being explicitly programmed. It works by using algorithms to analyze patterns, \n    make predictions, and solve problems based on the information provided. For example, machine learning is used

In [76]:
#fixed version of lcel
def query_rag_lcel(question):
    print(f"qustion: {question}")
    print("-"*50)

    #get source document separatly if needed
    answer = rag_chain_lcel.invoke(question)

    print(f"Answer: {answer}")

    docs = retriever.invoke(question)
    print("\nSource Document:")
    for i, doc in enumerate(docs):
        print(f"\n-- Source {i+1} ---")
        print(doc.page_content[:200] + "...")

In [78]:
#test lcel
print("Testing LCEL Chain:")
query_rag_lcel("what is machin learning?")

Testing LCEL Chain:
qustion: what is machin learning?
--------------------------------------------------
Answer: Machine learning is a branch of artificial intelligence that enables computers to learn from data and improve their performance without being explicitly programmed. It works by using algorithms to analyze patterns, make predictions, and solve problems based on the information provided. For example, it is used in recommendation systems like suggesting movies on streaming platforms or products on shopping websites.

Source Document:

-- Source 1 ---
Machin learning fundamentals

    Machine learning is a branch of artificial intelligence that enables computers to learn from data and 
    improve their performance without being explicitly programm...

-- Source 2 ---
Deep learning foundimantals

    Deep learning is a specialized branch of machine learning that uses artificial neural networks
    to process large amounts of data and solve complex problems. It is d...

-- Source